# Closed-Loop Gain Recovery Test

Synthetic closed-loop demonstration of the `newnucal` calibration pipeline:

1. Build a HERA-like array and a random chromatic sky model
2. Simulate visibilities with `ForwardModel` at several times (Earth rotation) and frequencies
3. Apply known per-frequency gain degeneracies (amplitude, phase, phase gradient)
4. Recover gains with the sky held fixed — verifying the solution reproduces the data
5. Show joint sky + gain recovery starting from a perturbed sky

The key physics being tested: the DPSS spectral constraints on the sky/beam model, combined with Earth rotation that decorrelates sky pixels from beam pixels across time, provide enough information to reduce the residuals.  Note that the recovered gains need not match the injected ones exactly — there are residual degeneracies — but the calibrated model visibilities should match the data.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from astropy.time import Time
from astropy.coordinates import EarthLocation
import healpy

jax.config.update("jax_enable_x64", False)  # float32 throughout

from newnucal import HERAArray, BeamModel, ForwardModel, Calibrator, apply_gains, init_gain_params
from newnucal.dpss import dpss_matrix
from newnucal.simulate import compute_rotation_matrices

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110})

## 1. Array, frequency, and time setup

In [ ]:

# --- Array ---
array = HERAArray.from_hex(hexnum=4, sep=14.6)
print(f"Antennas: {array.nants},  Baselines: {array.nbls}")

# --- Frequencies ---
nfreq = 64
freqs = np.linspace(50e6, 225e6, nfreq)  # Hz

# --- Times ---
from astropy.time import Time
import astropy.units as u

hera_loc = EarthLocation(lat=-30.7215 * u.deg, lon=21.4283 * u.deg, height=1073.0 * u.m)
t0 = Time("2023-03-21T04:00:00", scale="utc")
ntime = 8
dt = 60 / ntime * u.min
times = t0 + np.arange(ntime) * dt
print(f"Times: {ntime},  span: {(times[-1] - times[0]).to(u.min):.1f}")

# --- Rotation matrices ---
from newnucal.simulate import compute_rotation_matrices
rot_matrices = compute_rotation_matrices(times, hera_loc)
print(f"rot_matrices shape: {rot_matrices.shape}")


## 2. Beam and sky model

In [ ]:
sky_nside = 32
beam_nside = 16

# Beam: Airy disk (HERA dish diameter 14.6 m), DPSS eta_max = 20 ns
beam_model = BeamModel(nside=beam_nside, freqs=freqs, eta_max=20e-9)
print(f"Beam DPSS modes: {beam_model.A_beam.shape[1]}")

# Sky: random power-law HEALPix map
npix_sky = healpy.nside2npix(sky_nside)
rng = np.random.default_rng(42)

# Build a smooth chromatic sky: per-pixel spectral index drawn from N(-0.7, 0.1)
ref_freq = 150e6
spectral_indices = rng.normal(-0.7, 0.1, npix_sky).astype(np.float32)
ref_flux = rng.exponential(scale=1.0, size=npix_sky).astype(np.float32)
# flux_true[npix, nfreq]
flux_true = ref_flux[:, None] * (freqs[None, :] / ref_freq) ** spectral_indices[:, None]

# Project onto sky DPSS basis
sky_eta_max = 40e-9  # ns — wider than beam because sky has steeper spectral structure
from newnucal.dpss import dpss_matrix, dpss_project
A_sky = dpss_matrix(freqs, sky_eta_max)
print(f"Sky DPSS modes: {A_sky.shape[1]}")

sky_coeffs_true = jnp.array(dpss_project(flux_true, A_sky), dtype=jnp.float32)
print(f"sky_coeffs_true shape: {sky_coeffs_true.shape}")


## 3. Simulate true visibilities and apply known gain perturbations

In [ ]:
fwd = ForwardModel(array, sky_nside, beam_model, freqs, eps=1e-5)
fwd.set_sky_dpss(A_sky)

print("Simulating true visibilities (this compiles the JIT on first run)...")
vis_true = fwd.simulate(sky_coeffs_true, jnp.array(rot_matrices))
print(f"vis_true shape: {vis_true.shape},  dtype: {vis_true.dtype}")
print(f"Mean |vis|: {float(jnp.abs(vis_true).mean()):.4f}")

In [ ]:
# --- True gain perturbations ---
# log_amp: smooth ~5% amplitude variation across band
true_log_amp = 0.05 * np.cos(2 * np.pi * np.arange(nfreq) / nfreq).astype(np.float32)

# phase: smooth ~0.15 rad variation across band
true_phase = 0.15 * np.sin(2 * np.pi * np.arange(nfreq) / nfreq).astype(np.float32)

# phi: small phase gradients, ~1e-4 rad/m, smooth across band
true_phi = np.zeros((2, nfreq), dtype=np.float32)
true_phi[0] = 1e-4 * np.cos(2 * np.pi * np.arange(nfreq) / nfreq)  # East
true_phi[1] = 5e-5 * np.sin(2 * np.pi * np.arange(nfreq) / nfreq)  # North

true_log_amp_j = jnp.array(true_log_amp)
true_phase_j   = jnp.array(true_phase)
true_phi_j     = jnp.array(true_phi)

vis_data = apply_gains(vis_true, true_log_amp_j, true_phase_j, true_phi_j,
                       jnp.array(array.bls, dtype=jnp.float32))

print(f"vis_data shape: {vis_data.shape}")
print(f"RMS gain amplitude perturbation: {float(jnp.exp(jnp.array(true_log_amp_j)).std()):.4f}")

## 4. Stage 1: Recover gains with sky held fixed (true sky)

With the sky fixed at truth, the gain-recovery problem looks nonlinear because gains enter the MSE loss as `exp(log_amp) * exp(i·(phase + phi@bl))`.  But it can be solved **exactly in one pass** via two linear steps:

1. **Matched-filter projection** (linear in the complex gain): for each (freq, baseline), the optimal unconstrained gain is `g[f,b] = Σ_t data·conj(model) / Σ_t |model|²`

2. **Log-domain linear regression** (linear in the 4 real parameters): `log g[f,b] = log_amp[f] + i·(phase[f] + phi_x[f]·bl_x[b] + phi_y[f]·bl_y[b])`, fit per frequency with weighted lstsq over baselines.

`Calibrator.fit_gains_linear` implements this.

In [ ]:

cal = Calibrator(
    array=array,
    beam_model=beam_model,
    sky_nside=sky_nside,
    sky_eta_max=sky_eta_max,
    freqs=freqs,
    rot_matrices=rot_matrices,
    data=vis_data,
    eps=1e-5,
)

loss_init = cal.calc_loss({"sky_coeffs": sky_coeffs_true, **init_gain_params(nfreq)})

# Analytic linear solve: exact in one pass, no iterations needed.
# Step 1: per-baseline optimal gains via matched-filter projection (linear in gains).
# Step 2: log-domain weighted lstsq to fit (log_amp, phase, phi_x, phi_y) per freq.
print("Fitting gains (analytic linear solve)...")
gain_params_fit, loss_fit = cal.fit_gains_linear(sky_coeffs_true)

print(f"Loss before: {loss_init:.4e}")
print(f"Loss after:  {loss_fit:.4e}")
print(f"Reduction:   {loss_init / loss_fit:.1f}x")


## 5. Data reproduction: baseline spectra and loss

In [ ]:

freq_mhz = freqs / 1e6
bls_j    = jnp.array(array.bls, dtype=jnp.float32)

params_before = {"sky_coeffs": sky_coeffs_true, **init_gain_params(nfreq)}
params_after  = {"sky_coeffs": sky_coeffs_true, **gain_params_fit}
vis_before = cal.simulate(params_before)
vis_after  = cal.simulate(params_after)

# --- Gain recovery (near-exact with the linear solver) ---
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
fig.suptitle("Stage 1: Recovered vs true gains  (analytic linear solve)", fontsize=12)

gain_panels = [
    (axes[0, 0], np.array(gain_params_fit["log_amp"]), true_log_amp, "log_amp"),
    (axes[0, 1], np.array(gain_params_fit["phase"]),   true_phase,   "phase (rad)"),
    (axes[1, 0], np.array(gain_params_fit["phi"][0]),  true_phi[0],  "phi_E (rad/m)"),
    (axes[1, 1], np.array(gain_params_fit["phi"][1]),  true_phi[1],  "phi_N (rad/m)"),
]
for ax, rec, true_val, ylabel in gain_panels:
    ax.plot(freq_mhz, true_val, "k-",  lw=2,   label="True")
    ax.plot(freq_mhz, rec,      "r--", lw=1.5, label="Recovered")
    ax.set_ylabel(ylabel); ax.set_xlabel("Frequency (MHz)"); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

print("Gain recovery residuals (RMS):")
print(f"  log_amp: {float(jnp.sqrt(jnp.mean((gain_params_fit['log_amp'] - true_log_amp_j)**2))):.2e}")
print(f"  phase:   {float(jnp.sqrt(jnp.mean((gain_params_fit['phase']   - true_phase_j  )**2))):.2e}")
print(f"  phi_E:   {float(jnp.sqrt(jnp.mean((gain_params_fit['phi'][0]  - true_phi_j[0] )**2))):.2e}")
print(f"  phi_N:   {float(jnp.sqrt(jnp.mean((gain_params_fit['phi'][1]  - true_phi_j[1] )**2))):.2e}")

# --- Baseline spectra: log amplitude + angle ---
bl_indices = [0, array.nbls // 4, array.nbls // 2, array.nbls - 1]
t_show = 0

fig, axes = plt.subplots(len(bl_indices), 2, figsize=(12, 2.8 * len(bl_indices)), sharex=True)
fig.suptitle(
    f"Stage 1: Baseline spectra (t={t_show})  —  loss {loss_init:.2e} → {loss_fit:.2e}",
    fontsize=11,
)
for row, bi in enumerate(bl_indices):
    bl_len = float(jnp.linalg.norm(bls_j[bi, :2]))

    ax = axes[row, 0]
    ax.semilogy(freq_mhz, jnp.abs(vis_data)[t_show, :, bi],   "k-",  lw=1.5, label="Data")
    ax.semilogy(freq_mhz, jnp.abs(vis_data - vis_before)[t_show, :, bi], "b--", lw=1,   label="Before", alpha=0.7)
    ax.semilogy(freq_mhz, jnp.abs(vis_data - vis_after)[t_show, :, bi],  "r--", lw=1.5, label="After")
    ax.set_ylabel(f"bl {bi} ({bl_len:.0f}m)\n|V|")
    if row == 0: ax.legend(fontsize=8)
    if row == len(bl_indices) - 1: ax.set_xlabel("Frequency (MHz)")

    ax = axes[row, 1]
    ax.plot(freq_mhz, jnp.angle(vis_data)[t_show, :, bi],   "k-",  lw=1.5, label="Data")
    ax.plot(freq_mhz, jnp.angle(vis_before)[t_show, :, bi], "b--", lw=1,   label="Before", alpha=0.7)
    ax.plot(freq_mhz, jnp.angle(vis_after)[t_show, :, bi],  "r--", lw=1.5, label="After")
    ax.set_ylabel(f"bl {bi} ({bl_len:.0f}m)\narg(V) (rad)")
    if row == 0: ax.legend(fontsize=8)
    if row == len(bl_indices) - 1: ax.set_xlabel("Frequency (MHz)")

plt.tight_layout(); plt.show()


## 6. Stage 2: Joint sky + gain recovery (perturbed sky start)

Start the sky from a slightly perturbed version of the truth (10% Gaussian noise on DPSS coefficients) and jointly optimize sky coefficients and gains. This tests whether Earth rotation + DPSS spectral constraints are sufficient to disentangle the two.

In [ ]:

# Perturb sky by 10% Gaussian noise on coefficients
rng2 = np.random.default_rng(99)
sky_coeffs_perturbed = sky_coeffs_true + 0.30 * jnp.array(
#sky_coeffs_perturbed = 0.50 * jnp.array(
    rng2.standard_normal(sky_coeffs_true.shape).astype(np.float32)
) * float(jnp.abs(sky_coeffs_true).mean())

params_joint0 = {
    "sky_coeffs": sky_coeffs_perturbed,
    **init_gain_params(nfreq),
}

#print("Joint sky+gain fit (L-BFGS)...")
#params_joint, loss_lbfgs = cal.fit_lbfgs(params_joint0, maxiter=60, tol=1e-7)
#print(f"  Loss: {loss_lbfgs:.4e}")

#print("Joint sky+gain fit (Adam)...")
#params_joint, loss_adam = cal.fit_optax(params_joint, maxiter=120, lr=1e-2, verbose=True)
#print(f"  Adam loss:   {loss_adam:.4e}")


In [ ]:
#gain_params, _ = cal.fit_gains_linear(sky_coeffs_perturbed)
#sky_coeffs, _ = cal.fit_sky_dirty(sky_coeffs_perturbed, gain_params, n_minor=1)
#sky_coeffs, gain_params, _ = cal.fit_alternating_dirty(sky_coeffs, params_joint, n_outer=3, n_minor=6, step_size=0.5, verbose=True)
#sky_coeffs, gain_params, _ = cal.fit_alternating_dirty(sky_coeffs, params_joint0, n_outer=3, n_minor=4, step_size=0.9, verbose=True)
sky_coeffs, gain_params, _ = cal.fit_alternating_dirty(sky_coeffs, gain_params, n_outer=3, n_minor=4, step_size=0.9, verbose=True)

In [ ]:
#gain_params, _ = cal.fit_gains_linear(sky_coeffs)
#sky_coeffs, _ = cal.fit_sky_dirty(sky_coeffs, gain_params, n_minor=1)

In [ ]:
params_joint = gain_params.copy()
params_joint['sky_coeffs'] = sky_coeffs

In [ ]:

vis_joint_before = cal.simulate(params_joint0)
vis_joint_after  = cal.simulate(params_joint)

loss_joint_before = cal.calc_loss(params_joint0)
loss_joint_after  = cal.calc_loss(params_joint)
print(f"Stage 2 loss — before: {loss_joint_before:.4e}   after: {loss_joint_after:.4e}")
print(f"  reduction: {loss_joint_before / loss_joint_after:.1f}x")

fig, axes = plt.subplots(len(bl_indices), 2, figsize=(12, 2.8 * len(bl_indices)), sharex=True)
fig.suptitle(
    f"Stage 2: Baseline spectra (t={t_show})  —  loss {loss_joint_before:.2e} → {loss_joint_after:.2e}",
    fontsize=11,
)
for row, bi in enumerate(bl_indices):
    bl_len = float(jnp.linalg.norm(bls_j[bi, :2]))

    ax = axes[row, 0]
    ax.semilogy(freq_mhz, jnp.abs(vis_data)[t_show, :, bi],         "k-",  lw=1.5, label="Data")
    ax.semilogy(freq_mhz, jnp.abs(vis_data - vis_joint_before)[t_show, :, bi], "b--", lw=1,   label="Before", alpha=0.7)
    ax.semilogy(freq_mhz, jnp.abs(vis_data - vis_joint_after)[t_show, :, bi],  "r--", lw=1.5, label="After")
    ax.set_ylabel(f"bl {bi} ({bl_len:.0f}m)\n|V|")
    if row == 0: ax.legend(fontsize=8)
    if row == len(bl_indices) - 1: ax.set_xlabel("Frequency (MHz)")

    ax = axes[row, 1]
    ax.plot(freq_mhz, jnp.angle(vis_data)[t_show, :, bi],         "k-",  lw=1.5, label="Data")
    ax.plot(freq_mhz, jnp.angle(vis_joint_before)[t_show, :, bi], "b--", lw=1,   label="Before", alpha=0.7)
    ax.plot(freq_mhz, jnp.angle(vis_joint_after)[t_show, :, bi],  "r--", lw=1.5, label="After")
    ax.set_ylabel(f"bl {bi} ({bl_len:.0f}m)\narg(V) (rad)")
    if row == 0: ax.legend(fontsize=8)
    if row == len(bl_indices) - 1: ax.set_xlabel("Frequency (MHz)")

plt.tight_layout(); plt.show()


## 7. Recovered sky map

In [ ]:

# Reconstruct sky flux from DPSS coefficients at a reference frequency
ifreq_ref = np.argmin(np.abs(freqs - ref_freq))

sky_true_map = np.array(flux_true[:, ifreq_ref])
sky_rec_map  = np.array(params_joint["sky_coeffs"] @ jnp.array(A_sky).T)[:, ifreq_ref]
sky_diff_map = sky_rec_map - sky_true_map

vmax = np.percentile(sky_true_map, 99)
vmin = 0.0

fig = plt.figure(figsize=(12, 9))
healpy.mollview(sky_true_map, fig=fig, sub=(3, 1, 1),
                title=f"True sky  ({ref_freq/1e6:.0f} MHz)", min=vmin, max=vmax,
                unit="Jy/pix", cmap="inferno")
healpy.mollview(sky_rec_map,  fig=fig, sub=(3, 1, 2),
                title="Recovered sky  (joint fit)", min=vmin, max=vmax,
                unit="Jy/pix", cmap="inferno")

diff_scale = np.percentile(np.abs(sky_diff_map), 99)
healpy.mollview(sky_diff_map, fig=fig, sub=(3, 1, 3),
                title="Residual (recovered − true)", min=-diff_scale, max=diff_scale,
                unit="Jy/pix", cmap="RdBu_r")
plt.show()

## Pixel-by-pixel scatter: recovered vs true
#fig, ax = plt.subplots(figsize=(5, 5))
#ax.scatter(sky_true_map, sky_rec_map, s=1, alpha=0.3, rasterized=True)
#lim = [vmin, vmax * 1.05]
#ax.plot(lim, lim, "k--", lw=1)
#ax.set_xlim(lim); ax.set_ylim(lim)
#ax.set_xlabel("True flux (Jy/pix)"); ax.set_ylabel("Recovered flux (Jy/pix)")
#ax.set_title(f"Sky pixel scatter  ({ref_freq/1e6:.0f} MHz)")
#plt.tight_layout(); plt.show()
#
#rms_err  = float(np.sqrt(np.mean(sky_diff_map**2)))
#rms_true = float(np.sqrt(np.mean(sky_true_map**2)))
#print(f"Sky RMS error / RMS true: {rms_err / rms_true:.3f}")
